<a href="https://colab.research.google.com/github/100522401/P1_AA_100522396_100522401/blob/main/notebooks/Cuaderno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica 1 NIA 100522396 NIA 100522401


# Introducción



El objetivo de la práctica es **construir un modelo** de aprendizaje automático capaz de predecir si un cliente bancario suscribirá un depósito a plazo. Para construir ese modelo nos basaremos en la información demográfica, financiera e interacciones de los clientes.

Como variable objetivo del problema tenemos 'deposit'. Esta variable indicará si el cliente contrató, o no, el producto ofrecido. Por tanto, se trata de un problema de **clasificación binaria**.
<br><br>
En este cuaderno se incluye:

  - Exploratory Data Analysis (EDA).
  - Definir una estrategia de evaluación.
  - Optimización de hiperparámetros (HPO).
  - Selección del mejor modelo y estudio de su rendimiento.
  - Entrenamiento del modelo final.

## Configuración inicial y carga de datos

En esta sección se importan las librerías básicas usadas, se fija la semilla con el nía 1005224**01** y se cargan los datos desde Google Drive

En primer lugar, se importan las librerías básicas y fijamos la semilla aleatoria.

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 1    #100522401
np.random.seed(SEED)

Cargamos el dataset de entrenamiento desde Google Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/P1_AA_datasets"
TRAIN_PATH = f"{BASE_DIR}/bank_01.pkl"
# COMP_PATH  = f"{BASE_DIR}/bank_competition.pkl"

print("TRAIN_PATH:", TRAIN_PATH)
# print("COMP_PATH:", COMP_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TRAIN_PATH: /content/drive/MyDrive/P1_AA_datasets/bank_01.pkl


Se comprueba que el dataset se ha cargado correctamente

In [12]:
#TODO: Se podrían añadir try/excepts al read...

data = pd.read_pickle(TRAIN_PATH)
display(data.head())
print("\nDatos cargados correctamente.")

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes



Datos cargados correctamente.


# EDA

En esta sección se realiza un análisis exploratorio de los datos (EDA) básico.

En este estudio determinaremos:

- Número de instancias.
- Número de variables ( incluyendo la variable objetivo).
- Identificación de variables numéricas, categóricas u ordinales.
- Estudio de cardinalidad.
- Análisis de la variable 'pdays'.
- Estudio de los valores faltantes, constantes o identificadores.
- Estudio del problema y análisis de desbalanceos de la variable objetivo.


In [13]:
#Tamaño del dataset y tipo de problema
n_rows, n_cols = data.shape
print(f"Instancias: {n_rows}")
print(f"Variables totales (incluyendo target): {n_cols}")

target = "deposit"
if target not in data.columns:
    raise ValueError(f"No existe la variable objetivo '{target}' en el dataset.")

n_unique_target = data[target].nunique(dropna=False)
print(f"Variable objetivo: {target}")
print(f"Valores únicos en target: {n_unique_target}")

# La variable 'deposit' indica si suscribe o no: problema de clasificación binaria.
if n_unique_target == 2:
    problem_type = "clasificación binaria"
else:
    problem_type = "clasificación (revisar codificación de clases)"

print(f"Tipo de problema: {problem_type}")

Instancias: 11000
Variables totales (incluyendo target): 17
Variable objetivo: deposit
Valores únicos en target: 2
Tipo de problema: clasificación binaria


In [14]:
# 2) Tipos de variables
X = data.drop(columns=[target])

num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numéricas: {len(num_cols)}")
print(num_cols)
print("\nCategóricas: {0}".format(len(cat_cols)))
print(cat_cols)

Numéricas: 7
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Categóricas: 9
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [15]:
# 3) Cardinalidad en variables categóricas
if len(cat_cols) == 0:
    print("No hay variables categóricas.")
else:
    cardinality = X[cat_cols].nunique(dropna=False).sort_values(ascending=False)
    display(cardinality.to_frame("n_unique"))

    high_card_cols = cardinality[cardinality > 10].index.tolist()
    if high_card_cols:
      print("\nCategóricas con alta cardinalidad (>10):", high_card_cols)
    else:
      print("\nCategóricas con alta cardinalidad (>10): ninguna")

,n_unique
job,13
month,12
education,5
poutcome,4
marital,3
contact,3
default,2
housing,2
loan,2



Categóricas con alta cardinalidad (>10): ['job', 'month']


In [16]:
# 4) Valores faltantes
missing_count = data.isna().sum().sort_values(ascending=False)
missing_ratio = (missing_count / len(data) * 100).round(2)
missing_df = pd.DataFrame({"n_missing": missing_count, "pct_missing": missing_ratio})
missing_df = missing_df[missing_df["n_missing"] > 0]

if missing_df.empty:
    print("No hay valores faltantes.")
else:
    display(missing_df)


,n_missing,pct_missing
education,316,2.87
job,169,1.54


In [18]:
# 5) Columnas constantes y posibles ID
nunique = X.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()

id_like_by_name = [c for c in X.columns if "id" in c.lower()]
id_like_by_uniqueness = [c for c in X.columns if X[c].nunique(dropna=False) >= 0.98 * len(X)]
id_like_cols = sorted(set(id_like_by_name + id_like_by_uniqueness))

print("Columnas constantes:", constant_cols if constant_cols else "ninguna")
print("Posibles columnas ID:", id_like_cols if id_like_cols else "ninguna")

age            76
job            12
marital         3
education       4
default         2
balance      3783
housing         2
loan            2
contact         3
day            31
month          12
duration     1423
campaign       36
pdays         472
previous       34
poutcome        4
dtype: int64
Columnas constantes: ninguna
Posibles columnas ID: ninguna


In [23]:
# Analisis de pdays
p = data["pdays"]

n_total = len(p)
n_minus1 = (p == -1).sum()
pct_minus1 = 100 * n_minus1 / n_total

print(f"Total: {n_total}")
print(f"pdays = -1 (sin contacto previo): {n_minus1} ({pct_minus1:.2f}%)")

p_valid = p[p != -1]
print("\nResumen pdays (solo contactos previos reales):")
n_notminus1 = (p != -1).sum()
pct_notminus1 = 100 * n_notminus1 / n_total
print(f"pdays != -1 (solo contactos previos reales): {n_notminus1} ({pct_notminus1:.2f}%)")

display(p_valid.describe())



Total: 11000
pdays = -1 (sin contacto previo): 8203 (74.57%)

Resumen pdays (solo contactos previos reales):
pdays != -1 (solo contactos previos reales): 2797 (25.43%)


,pdays
count,2797.000000
mean,204.718627
std,122.397953
min,1.000000
25%,98.000000
50%,182.000000
75%,286.000000
max,854.000000
